# Joint regularization pilot: term suppression in Gaussian additive models

This page reports a pilot that asks whether an extra penalty can remove whole smooth
terms from a Gaussian additive model. A spline term has two parts that a standard
penalty treats together: the curved part, which the usual smoothness penalty
controls, and the straight-line part, which it leaves unpenalized. The pilot adds
a second penalty on the straight-line part. A term can then shrink to zero effect
rather than to a line. The pilot compares this with the boosting procedure that mboostrs uses by default.

The protocol is `docs/benchmarks/joint_regularization/PROTOCOL.md`. The results are
in `results.json`. This page recomputes every table from that file.

## Design

Four scenarios, each with eight smooth predictors and 600 training rows. The noise
variance is 1. Each method is scored on 2000 fresh test rows, against the noiseless
conditional mean, so the error is the signal error only. Each scenario uses 20 seeds.

| Scenario | What it contains |
|---|---|
| Sparse smooth | a few predictors carry a smooth effect; the rest are null |
| Correlated substitute | predictors are strongly correlated, so several can stand in for one another |
| Dense smooth | all predictors carry a smooth effect |
| Null | no predictor has an effect |

The four methods:

- **Tuned boosting**: the library's default, stopped by cross-validation.
- **Selected-term REML**: the terms that boosting kept, refitted by REML (restricted maximum likelihood).
- **All-term REML**: every term, fitted jointly by REML with its ordinary penalty.
- **Double-penalty REML**: every term, fitted jointly by REML, with the second penalty
  on the straight-line part.

In [1]:
import json
from pathlib import Path

import polars as pl
from great_tables import GT

# This page runs from docs/book/benchmarks; the results file sits beside the study.
RESULTS = Path("evidence/joint_regularization/results.json")
data = json.loads(RESULTS.read_text())

rows = pl.DataFrame(data["rows"])
print(f"{rows.height} fits; statuses: {rows['status'].unique().to_list()}")

320 fits; statuses: ['ok']


The first check is that every fit completed. The check counts the rows and
lists their statuses, and counts the fits per method: 20 seeds for each of four
scenarios and four methods makes 320 fits.

In [2]:
rows.group_by("method").len().sort("method")

method,len
str,u32
"""double_penalty""",80
"""joint_reml""",80
"""selected_refit""",80
"""tuned_boost""",80


## Signal error by scenario and method

The table gives the mean signal mean squared error (MSE) over 20 seeds. Lower is
better. Bold marks the lowest value in each row. The standard error is the standard
deviation over seeds divided by the square root of 20.

In [3]:
methods = ["tuned_boost", "selected_refit", "joint_reml", "double_penalty"]
labels = {
    "tuned_boost": "Tuned boosting",
    "selected_refit": "Selected-term REML",
    "joint_reml": "All-term REML",
    "double_penalty": "Double-penalty REML",
}
scenarios = ["sparse", "correlated", "dense", "null"]
summary = data["summary"]

def cell(scenario, method, best):
    """Format one mean MSE with its standard error, bolding the lowest mean."""
    entry = summary[f"{scenario}/{method}"]
    text = f"{entry['mean_signal_mse']:.4f} ({entry['se_signal_mse']:.4f})"
    return f"**{text}**" if method == best else text

table_rows = []
for scenario in scenarios:
    means = {m: summary[f"{scenario}/{m}"]["mean_signal_mse"] for m in methods}
    best = min(means, key=means.get)
    entry = {"Scenario": scenario.capitalize()}
    for m in methods:
        entry[labels[m]] = cell(scenario, m, best)
    table_rows.append(entry)

mse_table = pl.DataFrame(table_rows)
(
    GT(mse_table)
    .tab_header(
        title="Signal MSE by scenario and method (mean over 20 seeds, SE in parentheses)",
        subtitle="Lower is better; 600 training rows; noise variance 1; error against the noiseless conditional mean",
    )
    .fmt_markdown(columns=list(labels.values()))
    .tab_stubhead(label="Scenario")
    .tab_source_note(
        "Bold: lowest mean in the row. SE is the seed-level standard deviation divided by sqrt(20). "
        "Source: summary in results.json."
    )
)

GT(_tbl_data=shape: (4, 5)
┌────────────┬─────────────────────┬─────────────────────┬─────────────────┬─────────────────────┐
│ Scenario   ┆ Tuned boosting      ┆ Selected-term REML  ┆ All-term REML   ┆ Double-penalty REML │
│ ---        ┆ ---                 ┆ ---                 ┆ ---             ┆ ---                 │
│ str        ┆ str                 ┆ str                 ┆ str             ┆ str                 │
╞════════════╪═════════════════════╪═════════════════════╪═════════════════╪═════════════════════╡
│ Sparse     ┆ 0.0298 (0.0018)     ┆ 0.0287 (0.0018)     ┆ 0.0294 (0.0018) ┆ **0.0219 (0.0016)** │
│ Correlated ┆ 0.0906 (0.0053)     ┆ 0.0473 (0.0036)     ┆ 0.0473 (0.0036) ┆ **0.0388 (0.0035)** │
│ Dense      ┆ 0.0462 (0.0019)     ┆ **0.0424 (0.0023)** ┆ 0.0424 (0.0023) ┆ 0.0599 (0.0047)     │
│ Null       ┆ **0.0031 (0.0007)** ┆ 0.0116 (0.0018)     ┆ 0.0181 (0.0017) ┆ 0.0067 (0.0010)     │
└────────────┴─────────────────────┴─────────────────────┴─────────────────┴─────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10971cce0>, _boxhead=Boxhead([ColInfo(var='Scenario', type=<ColInfoTypeEnum.default: 1>, column_label='Scenario', column_align='left', column_width=None), ColInfo(var='Tuned boosting', type=<ColInfoTypeEnum.default: 1>, column_label='Tuned boosting', column_align='left', column_width=None), ColInfo(var='Selected-term REML', type=<ColInfoTypeEnum.default: 1>, column_label='Selected-term REML', column_align='left', column_width=None), ColInfo(var='All-term REML', type=<ColInfoTypeEnum.default: 1>, column_label='All-term REML', column_align='left', column_width=None), ColInfo(var='Double-penalty REML', type=<ColInfoTypeEnum.default: 1>, column_label='Double-penalty REML', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10aa53d70>, _spanners=Spanners([]), _heading=Heading(title='Signal MSE by scenario and method (mean over 20 seeds, SE in parentheses)', subtitle='Lower is better; 600 training rows; noise variance 1; error against the noiseless conditional mean', preheader=None), _stubhead='Scenario', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10a1af560>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10a915310>, _source_notes=['Bold: lowest mean in the row. SE is the seed-level standard deviation divided by sqrt(20). Source: summary in results.json.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x109d25ee0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x109745fd0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#

The double-penalty fit has the lowest error on the sparse and correlated scenarios.
On the dense and null scenarios it has higher error than boosting. Boosting is best
on the null scenario. All-term REML is best on the dense scenario, where it ties
with selected-term REML.

## Paired comparison with boosting

The seeds reuse the same data for every method, so the difference between two methods
on one seed is less noisy than either error alone. The next cell takes the
double-penalty minus boosting difference for each seed and reports its mean, with the
standard error of that mean.

In [4]:
# Pair the two methods on each (scenario, seed) so the comparison uses the same data.
pairs = (
    rows.filter(pl.col("method").is_in(["tuned_boost", "double_penalty"]))
    .select(["scenario", "seed", "method", "signal_mse"])
    .pivot(on="method", index=["scenario", "seed"], values="signal_mse")
    .with_columns((pl.col("double_penalty") - pl.col("tuned_boost")).alias("difference"))
)

paired = (
    pairs.group_by("scenario", maintain_order=True)
    .agg(
        pl.col("difference").mean().alias("Mean difference"),
        (pl.col("difference").std() / pl.col("difference").count().sqrt()).alias("SE"),
        pl.len().alias("Seeds"),
    )
    .with_columns(pl.col("scenario").str.to_titlecase())
)
(
    GT(paired)
    .tab_header(
        title="Double-penalty REML minus tuned boosting, paired by seed",
        subtitle="Negative values favour double-penalty REML; positive values favour boosting",
    )
    .fmt_number(columns=["Mean difference", "SE"], decimals=5)
    .tab_stubhead(label="Scenario")
    .tab_source_note("SE is the standard deviation of the 20 paired differences divided by sqrt(20). Source: rows in results.json.")
)

GT(_tbl_data=shape: (4, 4)
┌────────────┬─────────────────┬──────────┬───────┐
│ scenario   ┆ Mean difference ┆ SE       ┆ Seeds │
│ ---        ┆ ---             ┆ ---      ┆ ---   │
│ str        ┆ f64             ┆ f64      ┆ u32   │
╞════════════╪═════════════════╪══════════╪═══════╡
│ Sparse     ┆ -0.007868       ┆ 0.001748 ┆ 20    │
│ Correlated ┆ -0.051856       ┆ 0.003263 ┆ 20    │
│ Dense      ┆ 0.013743        ┆ 0.003667 ┆ 20    │
│ Null       ┆ 0.003607        ┆ 0.000624 ┆ 20    │
└────────────┴─────────────────┴──────────┴───────┘, _body=<great_tables._gt_data.Body object at 0x109744a70>, _boxhead=Boxhead([ColInfo(var='scenario', type=<ColInfoTypeEnum.default: 1>, column_label='scenario', column_align='left', column_width=None), ColInfo(var='Mean difference', type=<ColInfoTypeEnum.default: 1>, column_label='Mean difference', column_align='right', column_width=None), ColInfo(var='SE', type=<ColInfoTypeEnum.default: 1>, column_label='SE', column_align='right', column_width=None), ColInfo(var='Seeds', type=<ColInfoTypeEnum.default: 1>, column_label='Seeds', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1099441a0>, _spanners=Spanners([]), _heading=Heading(title='Double-penalty REML minus tuned boosting, paired by seed', subtitle='Negative values favour double-penalty REML; positive values favour boosting', preheader=None), _stubhead='Scenario', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x109927dd0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x109927a10>, _source_notes=['SE is the standard deviation of the 20 paired differences divided by sqrt(20). Source: rows in results.json.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10aaa2c00>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10a9a4b30>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=F

The paired differences match the protocol's reported values: a gain of about 0.008 on
sparse smooth, about 0.052 on correlated substitute, and losses of about 0.014 on
dense smooth and about 0.004 on null. Each SE describes Monte Carlo precision within
one scenario and says nothing about other signal shapes.

## Computational cost

The timings are median wall-clock seconds per fit, including tuning where a method has
it. The selected-term refit is charged its full tuning cost plus the refit time.
Prediction and effect reporting are not included.

In [5]:
timing_rows = []
for scenario in scenarios:
    entry = {"Scenario": scenario.capitalize()}
    for m in methods:
        entry[labels[m]] = summary[f"{scenario}/{m}"]["median_wall_seconds"]
    timing_rows.append(entry)

timing = pl.DataFrame(timing_rows)
(
    GT(timing)
    .tab_header(
        title="Median fit time by scenario and method",
        subtitle="Seconds per fit on a shared machine, single thread; 600 training rows",
    )
    .fmt_number(columns=list(labels.values()), decimals=3)
    .tab_stubhead(label="Scenario")
    .tab_source_note("Descriptive timings, not a benchmark of the optimizer alone. Source: summary in results.json.")
)

GT(_tbl_data=shape: (4, 5)
┌────────────┬────────────────┬────────────────────┬───────────────┬─────────────────────┐
│ Scenario   ┆ Tuned boosting ┆ Selected-term REML ┆ All-term REML ┆ Double-penalty REML │
│ ---        ┆ ---            ┆ ---                ┆ ---           ┆ ---                 │
│ str        ┆ f64            ┆ f64                ┆ f64           ┆ f64                 │
╞════════════╪════════════════╪════════════════════╪═══════════════╪═════════════════════╡
│ Sparse     ┆ 0.014596       ┆ 0.023242           ┆ 0.013241      ┆ 0.021579            │
│ Correlated ┆ 0.057471       ┆ 0.071212           ┆ 0.014327      ┆ 0.024903            │
│ Dense      ┆ 0.026043       ┆ 0.041866           ┆ 0.011756      ┆ 0.018124            │
│ Null       ┆ 0.009486       ┆ 0.011025           ┆ 0.015087      ┆ 0.020247            │
└────────────┴────────────────┴────────────────────┴───────────────┴─────────────────────┘, _body=<great_tables._gt_data.Body object at 0x1099279b0>, _boxhead=Boxhead([ColInfo(var='Scenario', type=<ColInfoTypeEnum.default: 1>, column_label='Scenario', column_align='left', column_width=None), ColInfo(var='Tuned boosting', type=<ColInfoTypeEnum.default: 1>, column_label='Tuned boosting', column_align='right', column_width=None), ColInfo(var='Selected-term REML', type=<ColInfoTypeEnum.default: 1>, column_label='Selected-term REML', column_align='right', column_width=None), ColInfo(var='All-term REML', type=<ColInfoTypeEnum.default: 1>, column_label='All-term REML', column_align='right', column_width=None), ColInfo(var='Double-penalty REML', type=<ColInfoTypeEnum.default: 1>, column_label='Double-penalty REML', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10aaa2f30>, _spanners=Spanners([]), _heading=Heading(title='Median fit time by scenario and method', subtitle='Seconds per fit on a shared machine, single thread; 600 training rows', preheader=None), _stubhead='Scenario', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10a16cec0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10a009820>, _source_notes=['Descriptive timings, not a benchmark of the optimizer alone. Source: summary in results.json.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10aa52de0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10998b050>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', 

All-term REML is the fastest method on the sparse, correlated and dense scenarios.
Boosting is the fastest on the null scenario. The selected-term refit is slower than
boosting in every scenario, because it pays the full tuning cost before refitting.
These timings come from one shared machine, so read them as orders of magnitude.

## Findings

- **The extra penalty helps where a few terms carry the signal.** On the sparse
  scenario, double-penalty REML lowers the error by about 26% against boosting. On
  the correlated scenario it lowers the error by about 57%. Ordinary joint REML already
  captures most of the correlated gain.
- **It hurts where every term carries signal, and where none does.** On the dense
  scenario its error is about 30% higher than boosting's. On the null scenario it is
  about 2 times boosting's error.
- **The pilot fails the across-scenario accuracy gate that its protocol prespecified** (the pass rule that requires the gains on some scenarios to outweigh the losses on the others).
  The gain on two scenarios does not outweigh the losses on the other two, so boosting
  stays the default.
- **Refitting after boosting selects can increase error.** On the null scenario,
  selected-term REML has about 3.8 times boosting's error. On the sparse, correlated
  and dense scenarios it has lower error than boosting.

## Limits

- **Four scenarios, eight predictors, 600 rows.** The pilot does not establish how the
  methods scale to more predictors or more rows.
- **No interval coverage.** The study measures prediction error and cost. It does not
  test confidence or interval coverage.
- **Descriptive term counts.** Effective degrees of freedom (a measure of a term's flexibility) below 0.1 count as suppressed terms. That is a descriptive cutoff, not a support-recovery test.
- **Twenty seeds per scenario.** The seeds reuse the same generator families. A
  confirmation needs different signal shapes and correlation structures.
- **The recommendation for the next step.** The protocol recommends choosing among the
  three estimators on training data only, with held-out evaluation on new families.
  That experiment is not part of this pilot.

## Reproduction

The runner and build identity are recorded in `results.json`. From the repository root:

```sh
RAYON_NUM_THREADS=1 OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1 \
  PYTHONPATH=python:. .venv/bin/python -m benches.probe_joint_regularization \
  --out docs/benchmarks/joint_regularization/results.json
```